# Enhanced S3 to COG Converter with Chunked Processing

This notebook converts TIF files from S3 to Cloud Optimized GeoTIFFs (COGs) with:
- **Chunked processing** for memory-efficient handling of large files
- **Automatic AWS credential detection** (no .env file needed)
- **Download caching** to avoid re-downloading large files
- **COG validation** before uploading
- **Memory monitoring** and progress tracking

Author: Kyle Lesinger (Enhanced chunked version)

In [1]:
import os
import pandas as pd
import json
import tempfile
import boto3
import rasterio
from rasterio.windows import Window
from rasterio.enums import Resampling
from rasterio.warp import calculate_default_transform, reproject
from rasterio.io import MemoryFile
import rioxarray as rxr
import s3fs
import fsspec
from botocore.exceptions import NoCredentialsError, ClientError
from pathlib import Path
from datetime import datetime
import time
import numpy as np
import gc
import psutil
from tqdm import tqdm

print("✅ Libraries imported successfully!")
print(f"Boto3 version: {boto3.__version__}")
print(f"Rasterio version: {rasterio.__version__}")

✅ Libraries imported successfully!
Boto3 version: 1.39.11
Rasterio version: 1.4.3


In [2]:
# Add path for importing custom modules
import sys
from pathlib import Path

# Add the scripts directory to the Python path
scripts_dir = Path('../scripts').resolve()
if str(scripts_dir) not in sys.path:
    sys.path.insert(0, str(scripts_dir))

# Import functions from list_s3crawler_files module
from list_s3crawler_files import (
    load_drcs_data,
    get_tif_files_from_path,
    get_files_with_full_paths,
    list_available_directories
)

# Import COG and cache utilities
from cog_utilities import (
    check_cache_status,
    clear_cache,
    validate_cog,
    export_COG_PROFILE
)

# Import AWS S3 utilities
from aws_s3_utils import (
    initialize_s3_client,
    verify_s3_client,
    get_all_s3_keys
)

# Import batch processing utilities
from batch_processing import (
    process_file_batch,
    print_batch_summary
)

from memory_utils import (
    get_memory_usage,
    calculate_optimal_chunk_size,
    estimate_chunk_memory,
    format_bytes

)

from convert_utilities import (
    convert_to_proper_CRS_and_cogify_chunked
)
    
print("✅ Custom modules imported successfully!")
print(f"   Module path: {scripts_dir}")

✅ Memory monitoring utilities loaded
✅ Custom modules imported successfully!
   Module path: /home/jovyan/conversion_scripts/convert-files-and-move/scripts


# Memory Monitoring Utilities

# Useful links
<a href="https://data.disasters.openveda.cloud/browseui/browseui/#drcs_activations/" target="_blank" rel="noopener noreferrer" style="color: blue; font-size: 20px;">drcs_activations OLD Directory</a> -- You can view old directory file structure here.

<a href="https://docs.openveda.cloud/user-guide/content-curation/dataset-ingestion/file-preparation.html" target="_blank" rel="noopener noreferrer" style="color: blue; font-size: 20px;">VEDA docs for file naming conventions</a> -- Helps for understanding why/how we name content.

## List of new 2nd level directories

    "Sentinel-1"
    "Sentinel-2"
    "Landsat"
    "MODIS"
    "VIIRS"
    "ASTER"
    "MASTER"
    "ECOSTRESS"
    "Planet"
    "Maxar"
    "HLS"
    "IMERG"
    "GOES"
    "SMAP"
    "ICESat"
    "GEDI"
    "COMSAR"
    "UAVSAR"
    "WB-57"

In [3]:
# DO NOT CHANGE
DIR_OLD_BASE = 'drcs_activations'
DIR_NEW_BASE = 'drcs_activations_new'
BUCKET = 'nasa-disasters'

In [4]:

EVENT_NAME = '202410_Hurricane_Milton'  #find the name within drcs_activations OLD Directory (see link above)
PRODUCT_NAME = 'sentinel1'      #find the name within drcs_activations OLD Directory (see link above)
PATH_OLD = f'{DIR_OLD_BASE}/{EVENT_NAME}/{PRODUCT_NAME}'  # Updated to use actual available directory

In [5]:
# Define COG profile for rasterio (DO NOT CHANGE)
COG_PROFILE = export_COG_PROFILE()

# Chunked processing configuration
CHUNK_CONFIG = {
    "default_chunk_size": 1024,  # Default chunk size in pixels
    "memory_limit_mb": 500,      # Memory limit per chunk in MB
    "show_progress": True,       # Show progress bars
    "enable_memory_monitoring": True  # Monitor memory usage
}

## Initialize AWS S3 Client with automatic credential detection

In [6]:
# Initialize AWS S3 Client using the imported function
s3_client, fs_read = initialize_s3_client(bucket_name=BUCKET, verbose=True)

# Verify S3 client is ready using the imported function
verify_s3_client(s3_client, bucket_name=BUCKET, verbose=True)

# Get all TIF files using the imported function
keys = get_all_s3_keys(s3_client, BUCKET, PATH_OLD, ".tif") if s3_client else []

if keys:
    print(f"✅ Found {len(keys)} .tif files in the S3 bucket.")
else:
    print("No keys found or S3 client not initialized")
    
keys

⚠️ S3 client initialized (limited bucket list access)
✅ Confirmed access to nasa-disasters bucket
✅ S3 filesystem (fsspec) initialized
✅ S3 client ready for operations
   Bucket: nasa-disasters
   Ready to process files
✅ Found 22 .tif files in the S3 bucket.


['drcs_activations/202410_Hurricane_Milton/sentinel1/S1A_20241008_rgb.tif',
 'drcs_activations/202410_Hurricane_Milton/sentinel1/S1A_IW_20241003T232756_DVR_RTC20_G_gpuned_DF85_rgb.tif',
 'drcs_activations/202410_Hurricane_Milton/sentinel1/S1A_IW_20241003T232821_DVR_RTC20_G_gpuned_09AC_rgb.tif',
 'drcs_activations/202410_Hurricane_Milton/sentinel1/S1A_IW_20241003T232846_DVR_RTC20_G_gpuned_D437_rgb.tif',
 'drcs_activations/202410_Hurricane_Milton/sentinel1/S1A_IW_20241003T232911_DVR_RTC20_G_gpuned_9258_rgb.tif',
 'drcs_activations/202410_Hurricane_Milton/sentinel1/S1A_IW_20241003T232936_DVR_RTC20_G_gpuned_86E1_rgb.tif',
 'drcs_activations/202410_Hurricane_Milton/sentinel1/S1A_IW_20241003T233001_DVR_RTC20_G_gpuned_0314_rgb.tif',
 'drcs_activations/202410_Hurricane_Milton/sentinel1/S1A_IW_20241003T233026_DVR_RTC20_G_gpuned_D68D_rgb.tif',
 'drcs_activations/202410_Hurricane_Milton/sentinel1/S1A_IW_20241003T233051_DVR_RTC20_G_gpuned_F2F4_rgb.tif',
 'drcs_activations/202410_Hurricane_Milton/s

# For these we can see three different types of files

We will use the same rename function and place them into the same directory


## Configure bucket and paths (no need to create session manually)

In [7]:
def return_bucket_info(config):
    """
    Extract bucket information from configuration and return as dictionary.
    
    Args:
        config: Configuration dictionary containing bucket and prefix information
    
    Returns:
        Dictionary with bucket and prefix information
    """
    # Configure bucket and paths (no need to create session manually)
    bucket_name = config["cog_data_bucket"]
    raw_data_bucket = config["raw_data_bucket"]
    raw_data_prefix = config["raw_data_prefix"]
    
    cog_data_bucket = config['cog_data_bucket']
    cog_data_prefix = config["cog_data_prefix"]
    
    print(f"Configuration loaded:")
    print(f"  Source bucket: {raw_data_bucket}")
    print(f"  Source prefix: {raw_data_prefix}")
    print(f"  Target bucket: {cog_data_bucket}")
    print(f"  Target prefix: {cog_data_prefix}")

    return {
        "bucket_name": bucket_name,
        "raw_data_bucket": raw_data_bucket,
        "raw_data_prefix": raw_data_prefix,
        "cog_data_bucket": cog_data_bucket,
        "cog_data_prefix": cog_data_prefix
    }

## Define Chunked COG Conversion Function

This function handles the conversion of files to Cloud Optimized GeoTIFFs with:
- Chunked processing to handle large files
- Memory monitoring
- Progress tracking
- Proper CRS and caching

In [8]:
# Check current cache status using the imported function
check_cache_status()

📊 Cache Status:
  - Directory: data_download/
  - Total files: 33
  - Total size: 7.50 GB

📁 Cached files (first 10):
  - drcs_activations/202410_Hurricane_Milton/landsat/LC08_L1_colorInfrared_20241012_154915_015040.tif (168.4 MB)
  - drcs_activations/202410_Hurricane_Milton/landsat/LC08_L1_colorInfrared_20241012_154939_015041.tif (168.6 MB)
  - drcs_activations/202410_Hurricane_Milton/landsat/LC08_L1_colorInfrared_20241012_15503_015042.tif (169.1 MB)
  - drcs_activations/202410_Hurricane_Milton/landsat/LC08_L1_naturalColor_20241012_154915_015040.tif (168.4 MB)
  - drcs_activations/202410_Hurricane_Milton/landsat/LC08_L1_naturalColor_20241012_154939_015041.tif (168.6 MB)
  - drcs_activations/202410_Hurricane_Milton/landsat/LC08_L1_naturalColor_20241012_15503_015042.tif (169.1 MB)
  - drcs_activations/202410_Hurricane_Milton/landsat/LC08_L1_trueColor_20241012_154915_015040.tif (168.4 MB)
  - drcs_activations/202410_Hurricane_Milton/landsat/LC08_L1_trueColor_20241012_154939_015041.tif (1

(33, 8057599277)

In [9]:
import re

def simple_process_files(keys, filter_str, rename_func, target_dir, EVENT_NAME):
    """
    Simple wrapper to process files with minimal code.
    
    Args:
        keys: List of all S3 keys
        filter_str: Can be:
            - String to filter files (e.g. 'S1_WTR')
            - Regex pattern object (e.g. re.compile(r'.*S2A.*mosaic'))
            - Callable function that returns True/False
        rename_func: Your custom rename function
        target_dir: Target directory (e.g. "Sentinel-1/opera_dswx")
        EVENT_NAME: Event name
    
    Returns:
        Processing results DataFrame
    """
    # 1. Filter files based on type of filter_str
    if callable(filter_str):
        # If it's a function
        filtered_files = [i for i in keys if filter_str(i)]
    elif hasattr(filter_str, 'search'):
        # If it's a compiled regex pattern
        filtered_files = [i for i in keys if filter_str.search(i)]
    elif isinstance(filter_str, str) and filter_str.startswith('r"') or filter_str.startswith("r'"):
        # If it's a regex string (e.g., r'pattern')
        pattern = re.compile(filter_str[2:-1])  # Remove r" or r'
        filtered_files = [i for i in keys if pattern.search(i)]
    else:
        # Default: simple string contains
        filtered_files = [i for i in keys if filter_str in i]
    
    # 2. Test renaming
    print(f"Testing filenames:")
    for f in filtered_files:
        print(f"  {rename_func(f, EVENT_NAME)}")
    
    # 3. Setup config
    config = {
        "data_acquisition_method": "s3",
        "raw_data_bucket": BUCKET,
        "raw_data_prefix": PATH_OLD,
        "cog_data_bucket": BUCKET,
        "cog_data_prefix": f'{DIR_NEW_BASE}/{target_dir}',
        "local_output_dir": f"output/{EVENT_NAME}",
        "transformation": {}
    }
    return_bucket_info(config)
    
    # 4. Process files
    print("\n" + "="*50)
    print("🌊 Processing Files (Chunked)")
    print("="*50)
    
    def chunked_converter(name, BUCKET, cog_filename, cog_data_bucket, cog_data_prefix, s3_client, local_output_dir=None):
        return convert_to_proper_CRS_and_cogify_chunked(
            name, BUCKET, cog_filename, cog_data_bucket, cog_data_prefix, s3_client, COG_PROFILE,
            local_output_dir, chunk_config=CHUNK_CONFIG
        )

    results = process_file_batch(
        file_list=filtered_files,
        s3_client=s3_client,
        config=config,
        filename_creator_func=rename_func,
        processing_func=chunked_converter,
        event_name=EVENT_NAME,
        save_metadata=True,
        save_csv=True,
        verbose=True,
        BUCKET=BUCKET
    )
    
    print_batch_summary(results)
    return results

# Process files

In [10]:
keys

['drcs_activations/202410_Hurricane_Milton/sentinel1/S1A_20241008_rgb.tif',
 'drcs_activations/202410_Hurricane_Milton/sentinel1/S1A_IW_20241003T232756_DVR_RTC20_G_gpuned_DF85_rgb.tif',
 'drcs_activations/202410_Hurricane_Milton/sentinel1/S1A_IW_20241003T232821_DVR_RTC20_G_gpuned_09AC_rgb.tif',
 'drcs_activations/202410_Hurricane_Milton/sentinel1/S1A_IW_20241003T232846_DVR_RTC20_G_gpuned_D437_rgb.tif',
 'drcs_activations/202410_Hurricane_Milton/sentinel1/S1A_IW_20241003T232911_DVR_RTC20_G_gpuned_9258_rgb.tif',
 'drcs_activations/202410_Hurricane_Milton/sentinel1/S1A_IW_20241003T232936_DVR_RTC20_G_gpuned_86E1_rgb.tif',
 'drcs_activations/202410_Hurricane_Milton/sentinel1/S1A_IW_20241003T233001_DVR_RTC20_G_gpuned_0314_rgb.tif',
 'drcs_activations/202410_Hurricane_Milton/sentinel1/S1A_IW_20241003T233026_DVR_RTC20_G_gpuned_D68D_rgb.tif',
 'drcs_activations/202410_Hurricane_Milton/sentinel1/S1A_IW_20241003T233051_DVR_RTC20_G_gpuned_F2F4_rgb.tif',
 'drcs_activations/202410_Hurricane_Milton/s

In [12]:
def create_cog_filename_rgb(f, EVENT_NAME):
    """Create COG filename for ARIA DPM files, moving event name first and timestamp to end."""
    filename = Path(f).stem
    
    # First check if it's the simple format: S1A_YYYYMMDD_rgb
    simple_pattern = r'^(S1[AB])_(\d{8})_(rgb)$'
    simple_match = re.match(simple_pattern, filename)
    
    if simple_match:
        satellite = simple_match.group(1)
        date_str = simple_match.group(2)
        product_type = simple_match.group(3)
        
        # Format as date only (since no time is provided)
        formatted_date = f"{date_str[:4]}-{date_str[4:6]}-{date_str[6:8]}"
        cog_filename = f'{EVENT_NAME}_{satellite}_{product_type}_{formatted_date}_day.tif'
        return cog_filename
    
    # Otherwise, use the original logic for full timestamp format
    parts = filename.split('_')
    
    # Find the part with the timestamp (format: YYYYMMDDTHHMMSS)
    timestamp_part = None
    timestamp_index = None
    for i, part in enumerate(parts):
        if 'T' in part and len(part) == 15:  # YYYYMMDDTHHMMSS
            timestamp_part = part
            timestamp_index = i
            break
    
    if timestamp_part:
        # Parse the timestamp
        date_part = timestamp_part[:8]  # 20230719
        time_part = timestamp_part[9:]  # 231439
        
        # Format as ISO 8601: YYYY-MM-DDTHH:MM:SSZ
        formatted_timestamp = f"{date_part[:4]}-{date_part[4:6]}-{date_part[6:8]}T{time_part[:2]}:{time_part[2:4]}:{time_part[4:6]}Z"
        
        # Remove the timestamp from the original parts
        remaining_parts = parts[:timestamp_index] + parts[timestamp_index+1:]
        
        # Create new filename: EVENT_NAME_remaining_parts_timestamp_day.tif
        cog_filename = f'{EVENT_NAME}_{"_".join(remaining_parts)}_{formatted_timestamp}.tif'
    else:
        # Fallback if no timestamp found
        cog_filename = f'{EVENT_NAME}_{filename}.tif'
    
    return cog_filename

    
filter_str = 'rgb'

# Test functions
print("Testing WM filename:")
filter_ = [i for i in keys if filter_str in i]

for idx,i in enumerate(filter_):
    test_wm = create_cog_filename_rgb(filter_[idx], EVENT_NAME)
    print(f"  {test_wm}")




Testing WM filename:
  202410_Hurricane_Milton_S1A_rgb_2024-10-08_day.tif
  202410_Hurricane_Milton_S1A_IW_DVR_RTC20_G_gpuned_DF85_rgb_2024-10-03T23:27:56Z.tif
  202410_Hurricane_Milton_S1A_IW_DVR_RTC20_G_gpuned_09AC_rgb_2024-10-03T23:28:21Z.tif
  202410_Hurricane_Milton_S1A_IW_DVR_RTC20_G_gpuned_D437_rgb_2024-10-03T23:28:46Z.tif
  202410_Hurricane_Milton_S1A_IW_DVR_RTC20_G_gpuned_9258_rgb_2024-10-03T23:29:11Z.tif
  202410_Hurricane_Milton_S1A_IW_DVR_RTC20_G_gpuned_86E1_rgb_2024-10-03T23:29:36Z.tif
  202410_Hurricane_Milton_S1A_IW_DVR_RTC20_G_gpuned_0314_rgb_2024-10-03T23:30:01Z.tif
  202410_Hurricane_Milton_S1A_IW_DVR_RTC20_G_gpuned_D68D_rgb_2024-10-03T23:30:26Z.tif
  202410_Hurricane_Milton_S1A_IW_DVR_RTC20_G_gpuned_F2F4_rgb_2024-10-03T23:30:51Z.tif
  202410_Hurricane_Milton_S1A_IW_DVR_RTC20_G_gpuned_EF50_rgb_2024-10-03T23:31:16Z.tif
  202410_Hurricane_Milton_S1A_IW_DVR_RTC20_G_gpufed_8E8B_rgb_2024-10-11T11:25:51Z.tif
  202410_Hurricane_Milton_S1A_IW_DVR_RTC20_G_gpufed_A625_rgb_2024-

In [ ]:
# Process S1 WTR files
results1 = simple_process_files(keys=keys, 
                                filter_str = filter_str, 
                                rename_func = create_cog_filename_rgb, 
                                target_dir = "Sentinel-1/rgb", 
                                EVENT_NAME = EVENT_NAME)


Testing filenames:
  202410_Hurricane_Milton_S1A_rgb_2024-10-08_day.tif
  202410_Hurricane_Milton_S1A_IW_DVR_RTC20_G_gpuned_DF85_rgb_2024-10-03T23:27:56Z.tif
  202410_Hurricane_Milton_S1A_IW_DVR_RTC20_G_gpuned_09AC_rgb_2024-10-03T23:28:21Z.tif
  202410_Hurricane_Milton_S1A_IW_DVR_RTC20_G_gpuned_D437_rgb_2024-10-03T23:28:46Z.tif
  202410_Hurricane_Milton_S1A_IW_DVR_RTC20_G_gpuned_9258_rgb_2024-10-03T23:29:11Z.tif
  202410_Hurricane_Milton_S1A_IW_DVR_RTC20_G_gpuned_86E1_rgb_2024-10-03T23:29:36Z.tif
  202410_Hurricane_Milton_S1A_IW_DVR_RTC20_G_gpuned_0314_rgb_2024-10-03T23:30:01Z.tif
  202410_Hurricane_Milton_S1A_IW_DVR_RTC20_G_gpuned_D68D_rgb_2024-10-03T23:30:26Z.tif
  202410_Hurricane_Milton_S1A_IW_DVR_RTC20_G_gpuned_F2F4_rgb_2024-10-03T23:30:51Z.tif
  202410_Hurricane_Milton_S1A_IW_DVR_RTC20_G_gpuned_EF50_rgb_2024-10-03T23:31:16Z.tif
  202410_Hurricane_Milton_S1A_IW_DVR_RTC20_G_gpufed_8E8B_rgb_2024-10-11T11:25:51Z.tif
  202410_Hurricane_Milton_S1A_IW_DVR_RTC20_G_gpufed_A625_rgb_2024-10

Band 1:   5%|▌         | 66/1300 [00:02<00:56, 22.01chunks/s]


   [MEMORY] High usage: 597.4 MB, forcing cleanup...


Band 1:   6%|▌         | 75/1300 [00:03<00:58, 21.04chunks/s]


   [MEMORY] High usage: 616.7 MB, forcing cleanup...


...

   [VERIFY] Checking reprojected data...
   [VERIFY] RGB file without nodata - all pixel values are valid
   [VERIFY] Band 1: min=1, max=255, center sample non-zero=1000000/1000000
            Estimated data coverage: 100.0% (from distributed samples)
   [VERIFY] Band 2: min=1, max=255, center sample non-zero=1000000/1000000
            Estimated data coverage: 100.0% (from distributed samples)
   [VERIFY] Band 3: min=1, max=255, center sample non-zero=1000000/1000000
            Estimated data coverage: 100.0% (from distributed samples)
   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: uint8
   [NODATA] Using nodata value 0 for uint8 data
   [PREDICTOR] Data type: uint8, using PREDICTOR=2
   [WRITE] Writing temporary GeoTIFF with chunked processing...


Reading input: /tmp/tmpf1_avuf1_temp.tif



   [CONVERT] Converting GeoTIFF to COG...
   [CONVERT] Using rio-cogeo for conversion...


Adding overviews...
Updating dataset tags...
Writing output to: /tmp/tmplcuclfvl.tif


   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/Sentinel-1/rgb/202410_Hurricane_Milton_S1A_IW_DVR_RTC20_G_gpuned_86E1_rgb_2024-10-03T23:29:36Z.tif
   [MEMORY] Final: 1920.9 MB (Change: +0.0 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202410_Hurricane_Milton_S1A_IW_DVR_RTC20_G_gpuned_86E1_rgb_2024-10-03T23:29:36Z.tif

[7/19] Processing: drcs_activations/202410_Hurricane_Milton/sentinel1/S1A_IW_20241003T233001_DVR_RTC20_G_gpuned_0314_rgb.tif
   Output filename: 202410_Hurricane_Milton_S1A_IW_DVR_RTC20_G_gpuned_0314_rgb_2024-10-03T23:30:01Z.tif
   [MEMORY] Initial: 1920.9 MB
   [DOWNLOAD] Downloading from S3...


In [12]:
keys

['drcs_activations/202410_Hurricane_Milton/sentinel1/S1A_20241008_rgb.tif',
 'drcs_activations/202410_Hurricane_Milton/sentinel1/S1A_IW_20241003T232756_DVR_RTC20_G_gpuned_DF85_rgb.tif',
 'drcs_activations/202410_Hurricane_Milton/sentinel1/S1A_IW_20241003T232821_DVR_RTC20_G_gpuned_09AC_rgb.tif',
 'drcs_activations/202410_Hurricane_Milton/sentinel1/S1A_IW_20241003T232846_DVR_RTC20_G_gpuned_D437_rgb.tif',
 'drcs_activations/202410_Hurricane_Milton/sentinel1/S1A_IW_20241003T232911_DVR_RTC20_G_gpuned_9258_rgb.tif',
 'drcs_activations/202410_Hurricane_Milton/sentinel1/S1A_IW_20241003T232936_DVR_RTC20_G_gpuned_86E1_rgb.tif',
 'drcs_activations/202410_Hurricane_Milton/sentinel1/S1A_IW_20241003T233001_DVR_RTC20_G_gpuned_0314_rgb.tif',
 'drcs_activations/202410_Hurricane_Milton/sentinel1/S1A_IW_20241003T233026_DVR_RTC20_G_gpuned_D68D_rgb.tif',
 'drcs_activations/202410_Hurricane_Milton/sentinel1/S1A_IW_20241003T233051_DVR_RTC20_G_gpuned_F2F4_rgb.tif',
 'drcs_activations/202410_Hurricane_Milton/s

In [13]:


filter_str = 'WM'

# Test functions
print("Testing WM filename:")
filter_ = [i for i in keys if filter_str in i]

for idx,i in enumerate(filter_):
    test_wm = create_cog_filename_rgb(filter_[idx], EVENT_NAME)
    print(f"  {test_wm}")




Testing WM filename:
  202410_Hurricane_Milton_S1A_IW_DVR_RTC20_G_gpufed_8E8B_reclassified_WM_2024-10-11T11:25:51Z_day.tif
  202410_Hurricane_Milton_S1A_IW_DVR_RTC20_G_gpufed_A625_reclassified_WM_2024-10-11T11:26:19Z_day.tif
  202410_Hurricane_Milton_S1A_IW_DVR_RTC20_G_gpufed_76F3_reclassified_WM_2024-10-11T11:26:44Z_day.tif


In [14]:
# Process S1 WTR files
results1 = simple_process_files(keys=keys, 
                                filter_str = filter_str, 
                                rename_func = create_cog_filename_rgb, 
                                target_dir = "Sentinel-1/WM", 
                                EVENT_NAME = EVENT_NAME)


Testing filenames:
  202410_Hurricane_Milton_S1A_IW_DVR_RTC20_G_gpufed_8E8B_reclassified_WM_2024-10-11T11:25:51Z_day.tif
  202410_Hurricane_Milton_S1A_IW_DVR_RTC20_G_gpufed_A625_reclassified_WM_2024-10-11T11:26:19Z_day.tif
  202410_Hurricane_Milton_S1A_IW_DVR_RTC20_G_gpufed_76F3_reclassified_WM_2024-10-11T11:26:44Z_day.tif
Configuration loaded:
  Source bucket: nasa-disasters
  Source prefix: drcs_activations/202410_Hurricane_Milton/sentinel1
  Target bucket: nasa-disasters
  Target prefix: drcs_activations_new/Sentinel-1/WM

🌊 Processing Files (Chunked)
✅ Local output directory ready: output/202410_Hurricane_Milton

[1/3] Processing: drcs_activations/202410_Hurricane_Milton/sentinel1/S1A_IW_20241011T112551_DVR_RTC20_G_gpufed_8E8B_reclassified_WM.tif
   Output filename: 202410_Hurricane_Milton_S1A_IW_DVR_RTC20_G_gpufed_8E8B_reclassified_WM_2024-10-11T11:25:51Z_day.tif
   [MEMORY] Initial: 293.8 MB
   [DOWNLOAD] Downloading from S3...
   [DOWNLOAD] ✅ Saved to cache
   [REPROJECT] Conver

   [VERIFY] Checking reprojected data...
   [VERIFY] Band 1: min=0, max=4, center sample non-zero=98296/1000000
            Estimated data coverage: 4.3% (from distributed samples)
   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: int8
   [NODATA] Using nodata value -128 for int8 data
   [PREDICTOR] Data type: int8, using PREDICTOR=2
   [WRITE] Writing temporary GeoTIFF with chunked processing...


Reading input: /tmp/tmpwkhkwgzc_temp.tif



   [CONVERT] Converting GeoTIFF to COG...
   [CONVERT] Using rio-cogeo for conversion...


Adding overviews...
Updating dataset tags...
Writing output to: /tmp/tmp1uxl0f6_.tif


...

Adding overviews...
Updating dataset tags...
Writing output to: /tmp/tmp6v2i_zqt.tif


   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/Sentinel-1/WM/202410_Hurricane_Milton_S1A_IW_DVR_RTC20_G_gpufed_A625_reclassified_WM_2024-10-11T11:26:19Z_day.tif
   [MEMORY] Final: 860.9 MB (Change: -15.6 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202410_Hurricane_Milton_S1A_IW_DVR_RTC20_G_gpufed_A625_reclassified_WM_2024-10-11T11:26:19Z_day.tif

[3/3] Processing: drcs_activations/202410_Hurricane_Milton/sentinel1/S1A_IW_20241011T112644_DVR_RTC20_G_gpufed_76F3_reclassified_WM.tif
   Output filename: 202410_Hurricane_Milton_S1A_IW_DVR_RTC20_G_gpufed_76F3_reclassified_WM_2024-10-11T11:26:44Z_day.tif
   [MEMORY] Initial: 860.9 MB
   [DOWNLOAD] Downloading from S3...
   [DOWNLOAD] ✅ Saved to cache
   [REPROJECT] Converting to EPSG:4326 using c

   [VERIFY] Checking reprojected data...
   [VERIFY] Band 1: min=0, max=4, center sample non-zero=29132/1000000
            Estimated data coverage: 0.9% (from distributed samples)
   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: int8
   [NODATA] Using nodata value -128 for int8 data
   [PREDICTOR] Data type: int8, using PREDICTOR=2
   [WRITE] Writing temporary GeoTIFF with chunked processing...
   [CONVERT] Converting GeoTIFF to COG...
   [CONVERT] Using rio-cogeo for conversion...


Reading input: /tmp/tmptvtls25a_temp.tif

Adding overviews...
Updating dataset tags...
Writing output to: /tmp/tmp1lahkv53.tif


   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/Sentinel-1/WM/202410_Hurricane_Milton_S1A_IW_DVR_RTC20_G_gpufed_76F3_reclassified_WM_2024-10-11T11:26:44Z_day.tif
   [MEMORY] Final: 895.2 MB (Change: +34.3 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202410_Hurricane_Milton_S1A_IW_DVR_RTC20_G_gpufed_76F3_reclassified_WM_2024-10-11T11:26:44Z_day.tif

✅ Batch processing complete: 3 files processed
📊 Uploaded metadata to s3://nasa-disasters/drcs_activations_new/Sentinel-1/WM/metadata.json
📝 Saved processing log to s3://nasa-disasters/drcs_activations_new/Sentinel-1/WM/files_converted.csv
📁 COGs saved locally to: output/202410_Hurricane_Milton

📊 BATCH PROCESSING SUMMARY
Total files processed: 3
Successful: 3
Failed: 0
Success rate: 100.0%
Timest

## Check STATUS of file conversion and upload

<a href="https://data.disasters.openveda.cloud/browseui/browseui/#drcs_activations_new/" target="_blank" rel="noopener noreferrer" style="color: blue; font-size: 20px;">Disasters Bucket</a> -- You can view that the files actually made it to their correct destination.

## Memory Usage Summary

You can check the final memory usage and cleanup

In [13]:
# Final memory cleanup and report
gc.collect()
final_memory = get_memory_usage()
print(f"\n📊 Memory Usage Summary:")
print(f"  Current memory usage: {final_memory:.1f} MB")
print(f"  Available memory: {psutil.virtual_memory().available / 1024 / 1024:.1f} MB")
print(f"  Memory percent used: {psutil.virtual_memory().percent:.1f}%")


📊 Memory Usage Summary:
  Current memory usage: 1195.5 MB
  Available memory: 27319.3 MB
  Memory percent used: 13.6%
